# House Prices: Two-Stage Model 

## Pipeline Overview

1. **Data Cleaning** — 4-strategy missing value handling, dtype fixes, logic contradictions
2. **Feature Engineering v2** — 236 features: sparse-split, ordinal encoding, log transforms, interactions
3. **Deviation Features** — 29 features encoding "why this house deviates from linear expectation"
4. **Stage 1: abess GIC** — Log-linear model with L0-best-subset selection (~32 features)
5. **Stage 2: XGBoost** — Predicts dollar residuals with regularization (alpha=0.5, lambda=10)
6. **Final Prediction** — `exp(Stage1_log + Stage2_log_correction)` or equivalently `Stage1_dollar + Stage2_dollar_correction`

## 1. Setup

In [ ]:
# Install abess (L0 best-subset selection)
!pip install -q abess

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from abess import LinearRegression as AbessLR
from xgboost import XGBRegressor

print("✓ All imports OK")

## 2. Load Data

In [ ]:
# Kaggle competition data path
DATA_PATH = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/"

train_raw = pd.read_csv(DATA_PATH + "train.csv")
test_raw  = pd.read_csv(DATA_PATH + "test.csv")

train_id = train_raw['Id']
test_id  = test_raw['Id']
y = train_raw['SalePrice']

X_train = train_raw.drop(columns=['Id', 'SalePrice'])
X_test  = test_raw.drop(columns=['Id'])

# Merge for unified cleaning
n_train = X_train.shape[0]
all_data = pd.concat([X_train, X_test], axis=0, ignore_index=True)

print(f"Train: {n_train} rows | Test: {X_test.shape[0]} rows | Features: {X_train.shape[1]}")
print(f"SalePrice: ${y.min():,.0f} ~ ${y.max():,.0f}, mean=${y.mean():,.0f}")

## 3. Data Cleaning

### 3.1 Dtype Fix & Logic Contradictions

In [ ]:
# ─── 3.1 Dtype: numeric codes → categorical ───
for col in ['MSSubClass', 'MoSold', 'YrSold']:
    all_data[col] = all_data[col].astype(str)

# ─── 3.2 MasVnrArea > 0 but Type missing → BrkFace ───
mask = (all_data['MasVnrArea'] > 0) & (all_data['MasVnrType'].isna())
all_data.loc[mask, 'MasVnrType'] = 'BrkFace'
print(f"MasVnr fix: {mask.sum()} rows")

# ─── 3.3 HasGarage flag ───
all_data['HasGarage'] = (~all_data['GarageType'].isna()).astype(int)
print("HasGarage created")

### 3.2 Missing Values — Type A: NA = None

In [ ]:
# Categorical: missing = feature doesn't exist
na_means_none = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'
]
for col in na_means_none:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna('None')
print(f"Filled {len(na_means_none)} cols: NA → 'None'")

### 3.3 Missing Values — Type B: No Basement/Garage → 0

In [ ]:
# No basement → basement numericals = 0
bsmt_num_cols = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath']
for col in bsmt_num_cols:
    no_bsmt = (all_data['BsmtQual'] == 'None') & (all_data[col].isna())
    all_data.loc[no_bsmt, col] = 0
    remaining = all_data[col].isna()
    if remaining.sum() > 0:
        all_data[col] = all_data[col].fillna(all_data[col].median())

# No garage → garage numericals = 0
for col in ['GarageCars', 'GarageArea']:
    no_garage = (all_data['GarageType'] == 'None') & (all_data[col].isna())
    all_data.loc[no_garage, col] = 0
    remaining = all_data[col].isna()
    if remaining.sum() > 0:
        all_data[col] = all_data[col].fillna(all_data[col].median())

# GarageYrBlt: no garage → 0; has garage but missing → YearBuilt
no_garage_yr = (all_data['HasGarage'] == 0) & (all_data['GarageYrBlt'].isna())
all_data.loc[no_garage_yr, 'GarageYrBlt'] = 0
has_garage_missing = (all_data['HasGarage'] == 1) & (all_data['GarageYrBlt'].isna())
all_data.loc[has_garage_missing, 'GarageYrBlt'] = all_data.loc[has_garage_missing, 'YearBuilt']
print(f"GarageYrBlt: {no_garage_yr.sum()}→0, {has_garage_missing.sum()}→YearBuilt")

### 3.4 Missing Values — Type C: LotFrontage (KNN k=7)

In [ ]:
# KNN imputation for LotFrontage using related features
knn_features = ['LotArea', 'LotShape', 'LotConfig', 'Neighborhood', 'BldgType', 'HouseStyle', '1stFlrSF', 'YearBuilt']
knn_cat = ['LotShape', 'LotConfig', 'Neighborhood', 'BldgType', 'HouseStyle']
knn_num = ['LotArea', '1stFlrSF', 'YearBuilt']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat_encoded = ohe.fit_transform(all_data[knn_cat].astype(str))
scaler_knn = StandardScaler()
num_scaled = scaler_knn.fit_transform(all_data[knn_num])
X_knn = np.hstack([num_scaled, cat_encoded])

# Stack LotFrontage as first column for KNN imputation
lf_col = all_data['LotFrontage'].values.reshape(-1, 1)
full_knn = np.hstack([lf_col, X_knn])

imputer = KNNImputer(n_neighbors=7, weights='distance')
full_imputed = imputer.fit_transform(full_knn)
n_lf_miss = all_data['LotFrontage'].isnull().sum()
all_data['LotFrontage'] = full_imputed[:, 0]
print(f"LotFrontage: {n_lf_miss} missing → KNN k=7")

### 3.5 Missing Values — Type D: Tiny Gaps

In [ ]:
tiny_fills = {
    'Electrical': 'SBrkr', 'MSZoning': 'RL', 'MasVnrType': 'None',
    'MasVnrArea': 0, 'KitchenQual': 'TA', 'Functional': 'Typ',
    'SaleType': 'WD', 'Exterior1st': 'VinylSd', 'Exterior2nd': 'VinylSd',
}
for col, val in tiny_fills.items():
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(val)

remaining = all_data.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) > 0:
    print(f"⚠ {len(remaining)} cols still have missing values")
else:
    print("✓ All missing values resolved")

### 3.6 Drop Low-Variance & Split

In [ ]:
# Drop Utilities (near-constant) and PoolQC (replaced by PoolArea)
all_data.drop(columns=['Utilities', 'PoolQC'], inplace=True, errors='ignore')

# Split back to train/test
X_train_clean = all_data.iloc[:n_train].copy()
X_test_clean  = all_data.iloc[n_train:].copy()

# Save clean copies for deviation feature construction
clean_tr = X_train_clean.copy()
clean_te = X_test_clean.copy()

# Build SalePrice-bearing train for Feature Engineering
train_clean = X_train_clean.copy()
train_clean.insert(0, 'SalePrice', y.values)
train_clean.insert(0, 'Id', train_id.values)
test_clean = X_test_clean.copy()
test_clean.insert(0, 'Id', test_id.values)

print(f"Clean data: train={train_clean.shape}, test={test_clean.shape}")

## 4. Feature Engineering v2

### 4.0 Pre-compute Derived Features

In [ ]:
# Re-merge for unified FE processing
X_tr = train_clean.drop(columns=['Id', 'SalePrice'])
X_te = test_clean.drop(columns=['Id'])
all_fe = pd.concat([X_tr, X_te], axis=0, ignore_index=True)

# dtype fix (re-apply since CSV save/load loses types)
for col in ['MSSubClass', 'MoSold', 'YrSold']:
    all_fe[col] = all_fe[col].astype(str)

# ─── Derived features (MUST compute before column transformations) ───
all_fe['TotalSF'] = (all_fe['GrLivArea'] + all_fe['TotalBsmtSF']
                     + all_fe['WoodDeckSF'] + all_fe['OpenPorchSF'])
all_fe['TotalPorchSF'] = (all_fe['OpenPorchSF'] + all_fe['EnclosedPorch']
                          + all_fe['3SsnPorch'] + all_fe['ScreenPorch'])
all_fe['TotalBsmtFinSF'] = all_fe['BsmtFinSF1'] + all_fe['BsmtFinSF2']
all_fe['TotalBath'] = (all_fe['FullBath'] + 0.5*all_fe['HalfBath']
                       + all_fe['BsmtFullBath'] + 0.5*all_fe['BsmtHalfBath'])

# Age features
yr_sold = all_fe['YrSold'].astype(int)
yr_built = all_fe['YearBuilt'].astype(int)
yr_remod = all_fe['YearRemodAdd'].astype(int)
all_fe['HouseAge'] = yr_sold - yr_built
all_fe['RemodAge'] = yr_sold - yr_remod
all_fe['IsRemodeled'] = (yr_remod != yr_built).astype(int)
all_fe['IsNewHome'] = (yr_built == yr_sold).astype(int)
all_fe['Has2ndFloor'] = (all_fe['2ndFlrSF'] > 0).astype(int)

print(f"Pre-computed derived features, shape={all_fe.shape}")

### 4.1 Sparse Areas → HasX + LogXIfHas

In [ ]:
# Split sparse area features into existence + conditional log
sparse_candidates = [
    ('PoolArea', 95), ('MiscVal', 90), ('LowQualFinSF', 90),
    ('3SsnPorch', 90), ('ScreenPorch', 85), ('EnclosedPorch', 80),
    ('BsmtFinSF2', 80), ('MasVnrArea', 50), ('WoodDeckSF', 50),
    ('BsmtFinSF1', 40), ('OpenPorchSF', 30), ('BsmtUnfSF', 25),
]

for col, threshold in sparse_candidates:
    if col not in all_fe.columns:
        continue
    zero_pct = (all_fe[col] == 0).sum() / len(all_fe) * 100
    if zero_pct < threshold:
        continue  # not sparse enough → will be log-transformed instead
    all_fe[f'Has_{col}'] = (all_fe[col] > 0).astype(int)
    all_fe[f'Log_{col}_IfHas'] = np.where(all_fe[col] > 0, np.log(all_fe[col]), 0.0)
    all_fe.drop(columns=[col], inplace=True)

print(f"After sparse split: {all_fe.shape[1]} cols")

### 4.2 Continuous Areas → Log1p

In [ ]:
continuous_log = [
    'GrLivArea', 'LotArea', '1stFlrSF', 'LotFrontage',
    'TotalBsmtSF', '2ndFlrSF', 'GarageArea',
    'TotalSF', 'TotalPorchSF', 'TotalBsmtFinSF', 'GarageYrBlt',
]
for col in continuous_log:
    if col in all_fe.columns:
        all_fe[f'Log_{col}'] = np.log1p(all_fe[col])

print(f"After log transforms: {all_fe.shape[1]} cols")

### 4.3 Ordinal Qualities → Ordinal Encoding

In [ ]:
ordinal_mappings = {
    'ExterQual':    {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1},
    'ExterCond':    {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1},
    'BsmtQual':     {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0},
    'BsmtCond':     {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0},
    'HeatingQC':    {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1},
    'KitchenQual':  {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1},
    'FireplaceQu':  {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0},
    'GarageQual':   {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0},
    'GarageCond':   {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0},
    'BsmtExposure': {'Gd':4,'Av':3,'Mn':2,'No':1,'None':0},
    'BsmtFinType1': {'GLQ':6,'ALQ':5,'BLQ':4,'Rec':3,'LwQ':2,'Unf':1,'None':0},
    'BsmtFinType2': {'GLQ':6,'ALQ':5,'BLQ':4,'Rec':3,'LwQ':2,'Unf':1,'None':0},
    'GarageFinish': {'Fin':3,'RFn':2,'Unf':1,'None':0},
    'Fence':        {'GdPrv':4,'MnPrv':3,'GdWo':2,'MnWw':1,'None':0},
    'PavedDrive':   {'Y':3,'P':2,'N':1},
    'LotShape':     {'Reg':4,'IR1':3,'IR2':2,'IR3':1},
    'LandSlope':    {'Gtl':3,'Mod':2,'Sev':1},
    'Electrical':   {'SBrkr':5,'FuseA':4,'FuseF':3,'FuseP':2,'Mix':1},
    'Functional':   {'Typ':7,'Min1':6,'Min2':5,'Mod':4,'Maj1':3,'Maj2':2,'Sev':1,'Sal':0},
    'Street':       {'Pave':1,'Grvl':0},
    'Alley':        {'Pave':1,'Grvl':0,'None':0},
    'CentralAir':   {'Y':1,'N':0},
    'LandContour':  {'Lvl':4,'Bnk':3,'HLS':2,'Low':1},
}

for col, mapping in ordinal_mappings.items():
    if col in all_fe.columns:
        all_fe[f'{col}_Ord'] = all_fe[col].map(mapping).fillna(0).astype(int)
        all_fe.drop(columns=[col], inplace=True)

print(f"After ordinal encoding: {all_fe.shape[1]} cols")

### 4.4 Nominal Categories → OneHot

In [ ]:
cat_cols = all_fe.select_dtypes(include=['object']).columns.tolist()
print(f"Encoding {len(cat_cols)} categorical columns...")

for col in cat_cols:
    dummies = pd.get_dummies(all_fe[col], prefix=col, drop_first=True).astype(int)
    all_fe = pd.concat([all_fe, dummies], axis=1)

all_fe.drop(columns=cat_cols, inplace=True)
print(f"After OneHot: {all_fe.shape[1]} cols")

### 4.5 Interactions: TotalSF × Neighborhood

In [ ]:
# TotalSF × Neighborhood (each neighborhood gets its own slope with TotalSF)
nbhds = sorted(set(
    c.replace('Neighborhood_', '') for c in all_fe.columns
    if c.startswith('Neighborhood_')
))
for nbhd in nbhds:
    dummy_col = f'Neighborhood_{nbhd}'
    if dummy_col in all_fe.columns:
        all_fe[f'TotalSF_x_{nbhd}'] = all_fe['TotalSF'] * all_fe[dummy_col]

n_int = sum(1 for c in all_fe.columns if c.startswith('TotalSF_x_'))
print(f"TotalSF × Neighborhood: {n_int} interactions")

### 4.6 Remove Redundancy

In [ ]:
# ─── 6a: Drop raw columns, keep log versions ───
raw_to_drop = []
for col in continuous_log:
    log_col = f'Log_{col}'
    if col in all_fe.columns and log_col in all_fe.columns:
        raw_to_drop.append(col)
all_fe.drop(columns=raw_to_drop, inplace=True)
print(f"Dropped {len(raw_to_drop)} raw (keep log)")

# ─── 6b: Perfect collinearity (|r| > 0.999) ───
num_cols = all_fe.select_dtypes(include=[np.number]).columns.tolist()
temp_y_train = pd.Series(y.values, index=range(n_train))
to_drop_corr = set()
corr_matrix = all_fe[num_cols].iloc[:n_train].corr().abs()

for i in range(len(num_cols)):
    for j in range(i+1, len(num_cols)):
        if corr_matrix.iloc[i, j] > 0.999:
            c1, c2 = num_cols[i], num_cols[j]
            if c1 in to_drop_corr or c2 in to_drop_corr:
                continue
            r1 = abs(all_fe[c1].iloc[:n_train].corr(temp_y_train))
            r2 = abs(all_fe[c2].iloc[:n_train].corr(temp_y_train))
            to_drop_corr.add(c2 if r1 >= r2 else c1)

all_fe.drop(columns=list(to_drop_corr), inplace=True)
print(f"Dropped {len(to_drop_corr)} perfectly correlated")

# ─── 6c: Ultra-sparse OneHot (n < 5) ───
sparse_drop = []
for c in all_fe.select_dtypes(include=[np.number]).columns:
    if any(c.startswith(p) for p in ['Neighborhood_','MSZoning_','TotalSF_x_','MSSubClass_']):
        continue
    n_ones = (all_fe[c] == 1).sum()
    if 0 < n_ones < 5:
        sparse_drop.append(c)

all_fe.drop(columns=sparse_drop, inplace=True)
print(f"Dropped {len(sparse_drop)} ultra-sparse OneHot")

all_fe = all_fe.fillna(0)
print(f"Final feature count: {all_fe.shape[1]}")

### 4.7 Split to Train/Test Matrices

In [ ]:
# Split back
X_train_fe = all_fe.iloc[:n_train].copy()
X_test_fe  = all_fe.iloc[n_train:].copy()

X_base = X_train_fe.values.astype(np.float64)
X_test_base = X_test_fe.values.astype(np.float64)
feat_names = list(X_train_fe.columns)
y_dollar = y.values
y_log = np.log1p(y_dollar)
n = len(y_dollar)

print(f"Train FE matrix: {X_base.shape}")
print(f"Test FE matrix:  {X_test_base.shape}")
print(f"Target log(SalePrice): mean={y_log.mean():.3f}, std={y_log.std():.3f}")

## 5. Deviation Features (for Stage 2)

These 29 features encode "why does this house deviate from the linear model's expectation?"
— quality-to-size ratios, age-quality interactions, neighborhood deviation z-scores, etc.

In [ ]:
def build_deviation_features(clean_df, train_stats=None, is_train=True):
    """Build 29 deviation features from cleaned (pre-FE) data."""
    cf = clean_df
    feats = pd.DataFrame(index=cf.index)

    # Basic variables from raw clean data
    total_sf = (cf['GrLivArea'].fillna(0) + cf['TotalBsmtSF'].fillna(0)
                + cf['WoodDeckSF'].fillna(0) + cf['OpenPorchSF'].fillna(0))
    gr_liv = cf['GrLivArea'].fillna(0)
    garage_area = cf['GarageArea'].fillna(0)
    lot_area = cf['LotArea'].fillna(0)
    qual = cf['OverallQual'].fillna(5)
    cond = cf['OverallCond'].fillna(5)
    house_age = np.maximum(2010 - cf['YearBuilt'].fillna(1970), 0)
    tot_bath = (cf['FullBath'].fillna(0) + 0.5*cf['HalfBath'].fillna(0)
                + cf['BsmtFullBath'].fillna(0) + 0.5*cf['BsmtHalfBath'].fillna(0))
    tot_rooms = cf['TotRmsAbvGrd'].fillna(6)
    garage_cars = cf['GarageCars'].fillna(0)
    fireplaces = cf['Fireplaces'].fillna(0)
    bsmt_fin = cf['BsmtFinSF1'].fillna(0)
    bsmt_total = bsmt_fin + cf['BsmtUnfSF'].fillna(0)
    nbhd = cf['Neighborhood']

    # Group 1: Quality-to-size ratios
    feats['Dev_SF_per_Qual'] = total_sf / qual
    feats['Dev_GrLiv_per_Qual'] = gr_liv / qual
    feats['Dev_Garage_per_Qual'] = garage_area / qual
    feats['Dev_Bath_per_Qual'] = tot_bath / qual
    feats['Dev_Lot_per_Qual'] = lot_area / qual / 100

    # Group 2: Quality-age interaction
    feats['Dev_Qual_x_Age'] = qual * house_age / 100
    feats['Dev_Qual_over_Age'] = qual / np.maximum(house_age, 1)
    feats['Dev_OldHighQual'] = ((house_age >= 50) & (qual >= 7)).astype(float)
    feats['Dev_OldLowQual'] = ((house_age >= 50) & (qual <= 4)).astype(float)
    feats['Dev_NewLowQual'] = ((house_age <= 15) & (qual <= 5)).astype(float)
    feats['Dev_NewHighQual'] = ((house_age <= 15) & (qual >= 8)).astype(float)

    # Group 3: Neighborhood deviation
    if is_train:
        nbh_med_qual = nbhd.map(cf.groupby('Neighborhood')['OverallQual'].median())
        nbh_med_sf = nbhd.map(pd.Series(total_sf.values, index=cf.index).groupby(nbhd).median())
        nbh_med_age = nbhd.map(pd.Series(house_age.values, index=cf.index).groupby(nbhd).median())
    else:
        nbh_med_qual = nbhd.map(train_stats['nbh_med_qual'])
        nbh_med_sf = nbhd.map(train_stats['nbh_med_sf'])
        nbh_med_age = nbhd.map(train_stats['nbh_med_age'])

    feats['Dev_NbhQualGap'] = qual - nbh_med_qual.fillna(qual.median())
    feats['Dev_NbhSFGap'] = (total_sf - nbh_med_sf.fillna(total_sf.median())) / 100
    feats['Dev_NbhAgeGap'] = house_age - nbh_med_age.fillna(house_age.median())

    # Group 4: Density / ratio features
    feats['Dev_BathDensity'] = tot_bath / np.maximum(total_sf, 500) * 1000
    feats['Dev_GarageDensity'] = garage_area / np.maximum(total_sf, 500)
    feats['Dev_LotToLiving'] = lot_area / np.maximum(gr_liv, 500)
    feats['Dev_FinBsmtRatio'] = bsmt_fin / np.maximum(bsmt_total, 1)
    feats['Dev_RoomSize'] = gr_liv / np.maximum(tot_rooms, 1)
    feats['Dev_FireplaceDensity'] = fireplaces / np.maximum(total_sf, 500) * 1000

    # Group 5: Extreme combination flags
    feats['Dev_BigButLowQual'] = ((total_sf > total_sf.quantile(0.75)) & (qual <= 5)).astype(float)
    feats['Dev_SmallButHighQual'] = ((total_sf < total_sf.quantile(0.25)) & (qual >= 7)).astype(float)
    feats['Dev_GarageHeavy'] = ((garage_area > 800) & (total_sf < 2000)).astype(float)
    feats['Dev_LotRich'] = ((lot_area > 15000) & (total_sf < 1500)).astype(float)
    feats['Dev_QualityMismatch'] = np.abs(qual - cond)

    # Group 6: Quality × feature interactions (2nd order)
    feats['Dev_Qual_x_SF'] = qual * np.log1p(total_sf)
    feats['Dev_Qual_x_Bath'] = qual * tot_bath
    feats['Dev_Qual_x_Garage'] = qual * garage_cars
    feats['Dev_Qual_x_Fireplace'] = qual * fireplaces

    if is_train:
        stats = {
            'nbh_med_qual': cf.groupby('Neighborhood')['OverallQual'].median(),
            'nbh_med_sf': pd.Series(total_sf.values, index=cf.index).groupby(nbhd).median(),
            'nbh_med_age': pd.Series(house_age.values, index=cf.index).groupby(nbhd).median(),
        }
        return feats, stats
    return feats, None

# Build deviation features
dev_tr, dev_stats = build_deviation_features(clean_tr, is_train=True)
dev_te, _ = build_deviation_features(clean_te, train_stats=dev_stats, is_train=False)

print(f"Deviation features: {len(dev_tr.columns)} built")
print(f"Examples: {list(dev_tr.columns[:8])}...")

# Combine: 236 FE + 29 deviation = 265 features for Stage2
X_dev_tr = dev_tr.values.astype(np.float64)
X_dev_te = dev_te.values.astype(np.float64)
X_s2_train = np.column_stack([X_base, X_dev_tr])
X_s2_test  = np.column_stack([X_test_base, X_dev_te])
print(f"Stage2 input: {X_s2_train.shape[1]} features (236 FE + 29 deviation)")

## 6. Stage 1: abess GIC — Log-Linear Model

L0 best-subset selection with GIC criterion. Finds the ~32 features that have
the strongest *linear* relationship with log(SalePrice).

In [ ]:
# ─── Remove near-constant columns ───
stds = X_base.std(axis=0)
keep_mask = stds > 1e-10
X_all_k = X_base[:, keep_mask]
print(f"Removed {(~keep_mask).sum()} constant columns, {X_all_k.shape[1]} remain")

# ─── abess GIC: find optimal feature set ───
m_gic = AbessLR(support_size=range(5, min(150, X_all_k.shape[1])+1), ic_type='gic')
m_gic.fit(X_all_k, y_log)
gic_idx = np.where(m_gic.coef_ != 0)[0]
gic_feat_names = [feat_names[i] for i in np.where(keep_mask)[0][gic_idx]]
print(f"GIC selected {len(gic_idx)} features")
print(f"Top 10: {gic_feat_names[:10]}")

# ─── Full training on GIC features ───
X_gi_train = X_all_k[:, gic_idx]
X_gi_test  = X_test_base[:, keep_mask][:, gic_idx]

# Standardize
scaler_s1 = StandardScaler()
X_gi_train_s = scaler_s1.fit_transform(X_gi_train)
X_gi_test_s  = scaler_s1.transform(X_gi_test)

# Remove any constant columns after standardization
cm_s1 = X_gi_train_s.std(axis=0) > 1e-10
X_gi_train_f = X_gi_train_s[:, cm_s1]
X_gi_test_f  = X_gi_test_s[:, cm_s1]

# Final abess on cleaned GIC features
nf = X_gi_train_f.shape[1]
abess_s1 = AbessLR(support_size=range(1, min(100, nf)+1), ic_type='gic')
abess_s1.fit(X_gi_train_f, y_log)

# Predictions
lin_train_log = abess_s1.predict(X_gi_train_f)
lin_test_log  = abess_s1.predict(X_gi_test_f)

lin_train_dollar = np.expm1(lin_train_log)
lin_test_dollar  = np.expm1(lin_test_log)

print(f"Stage1 complete. Train predictions: ${lin_train_dollar.min():,.0f} ~ ${lin_train_dollar.max():,.0f}")
print(f"Test predictions: ${lin_test_dollar.min():,.0f} ~ ${lin_test_dollar.max():,.0f}")

## 7. Stage 2: XGBoost on Dollar Residuals

XGBoost predicts the *dollar residual* (y_true − ŷ_linear), using all 265 features
(236 FE + 29 deviation). Regularization (alpha=0.5, lambda=10) prevents overfitting
on the weak residual signal (SNR ≈ 0.11).

In [ ]:
# ─── Compute residuals ───
residual_dollar = y_dollar - lin_train_dollar
print(f"Residuals: mean=${residual_dollar.mean():,.0f}, std=${residual_dollar.std():,.0f}")
print(f"Range: ${residual_dollar.min():,.0f} ~ ${residual_dollar.max():,.0f}")

# ─── Train XGBoost with regularization ───
xgb_s2 = XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=1.0,
    reg_alpha=0.5,       # L1 regularization
    reg_lambda=10,        # L2 regularization
    random_state=42,
    verbosity=0,
)

xgb_s2.fit(X_s2_train, residual_dollar)

# ─── Predict residual correction ───
residual_test = xgb_s2.predict(X_s2_test)
residual_train = xgb_s2.predict(X_s2_train)

print(f"Stage2 complete.")
print(f"Train correction: ${residual_train.min():,.0f} ~ ${residual_train.max():,.0f}, mean=${residual_train.mean():,.0f}")
print(f"Test correction:  ${residual_test.min():,.0f} ~ ${residual_test.max():,.0f}, mean=${residual_test.mean():,.0f}")

# ─── Feature importance summary ───
imps = xgb_s2.feature_importances_
dev_start = X_base.shape[1]
dev_imps = imps[dev_start:]
fe_imps_total = imps[:dev_start].sum()
dev_imps_total = dev_imps.sum()
print(f"\nImportance: FE features={fe_imps_total*100:.1f}%, Deviation features={dev_imps_total*100:.1f}%")

## 8. Final Predictions & Submission

In [ ]:
# Final = Stage1 dollar prediction + Stage2 residual correction
pred_test_dollar = lin_test_dollar + residual_test

# Safety floor at $50,000
pred_test_dollar = np.maximum(pred_test_dollar, 50000)

print(f"Final predictions:")
print(f"  Range: ${pred_test_dollar.min():,.0f} ~ ${pred_test_dollar.max():,.0f}")
print(f"  Mean:  ${pred_test_dollar.mean():,.0f}")
print(f"  Median: ${np.median(pred_test_dollar):,.0f}")

# ─── Generate submission ───
submission = pd.DataFrame({
    'Id': test_id.values,
    'SalePrice': pred_test_dollar
})
submission.to_csv('submission.csv', index=False)
print(f"\n✓ submission.csv saved ({len(submission)} rows)")
print("Ready for Kaggle submission!")

## Model Summary

| Stage | Method | Features | Key Config |
|-------|--------|----------|------------|
| Stage 1 | abess GIC | 32 features (selected from 236) | Log-linear, L0 best-subset |
| Stage 2 | XGBoost | 265 features (236 FE + 29 deviation) | 800 trees, lr=0.05, depth=4, alpha=0.5, lambda=10 |
| Final | Linear + Residual correction | — | Dollar-space combination |
